# NCP Brain — Reservoir Baseline + eval_v0

Runs the reservoir baseline on a Colab GPU (T4) to establish the cost floor:
how well does a frozen random CfC core + trained readout perform?

Now also runs `run_specs` to validate the 5 behavioral predicates on the
random-init brain BEFORE training — every spec should fail at init.

Fetches code from the local repo's main branch (not PR branch).

Estimated runtime: ~2-5 minutes on T4.

In [ ]:
# ── setup ──────────────────────────────────────────────────────────────────
import sys, subprocess, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ncps", "torch", "numpy"], check=True)

import requests

REPO = "https://raw.githubusercontent.com/NavpreetST/helios/main"

os.makedirs("/content/aegis/brain", exist_ok=True)
os.makedirs("/content/aegis/eval", exist_ok=True)
os.makedirs("/content/aegis/observability", exist_ok=True)
os.makedirs("/content/aegis/nexus", exist_ok=True)

files = {
    "aegis/brain/ncp.py": f"{REPO}/aegis/brain/ncp.py",
    "aegis/brain/reservoir.py": f"{REPO}/aegis/brain/reservoir.py",
    "aegis/brain/trace.py": f"{REPO}/aegis/brain/trace.py",
    "aegis/eval/runner.py": f"{REPO}/aegis/eval/runner.py",
    "aegis/eval/spec_generator.py": f"{REPO}/aegis/eval/spec_generator.py",
    "aegis/eval/__init__.py": f"{REPO}/aegis/eval/__init__.py",
    "aegis/observability/paths.py": f"{REPO}/aegis/observability/paths.py",
    "aegis/nexus/neurobus.py": f"{REPO}/aegis/nexus/neurobus.py",
}

for path, url in files.items():
    os.makedirs(os.path.dirname(f"/content/{path}"), exist_ok=True)
    resp = requests.get(url)
    resp.raise_for_status()
    with open(f"/content/{path}", "w") as f:
        f.write(resp.text)
    print(f"  \u2713 {path}")

sys.path.insert(0, "/content")
print("\nSetup complete.")

In [ ]:
# ── Run reservoir baseline ────────────────────────────────────────────────
%run -m aegis.brain.reservoir --samples 4096 --seq-len 5 --epochs 200 --lr 1e-3

In [ ]:
# ── Run spec generator baseline on random-init brain ──────────────────────
# Every spec should FAIL on random init. If any passes, the spec is too loose.
import torch
from aegis.eval.spec_generator import SpecGenerator, EVAL_SEED_NAMESPACE
from aegis.eval.runner import run_specs

gen = SpecGenerator(seed=EVAL_SEED_NAMESPACE.start)  # eval seed

train_ds = gen.mixed(n_per_spec=256)
reports = run_specs(train_ds)

print("=== Random-init brain: spec results ===")
hard_gates = []
for r in reports:
    print(r.summary())
    if r.is_hard_gate:
        hard_gates.append(r)

print()
if hard_gates:
    print("=== Hard gates (ablation) ===")
    for r in hard_gates:
        print(r.summary())

print()
print("Expected: ALL specs FAIL (random init)")

In [ ]:
# ── Compositional holdout (eval-only) ─────────────────────────────────────
from aegis.eval.runner import run_spec

holdout = gen.compositional_holdout(n=128)
hr = run_spec(holdout)
print(f"Compositional holdout (threat \u2227 3am): {hr.summary()}")
print("Expected: FAIL (random init has no threat->urgency reflex)")

In [ ]:
# ── Optional: full BPTT comparison ───────────────────────────────────────
# Uncomment below to compare reservoir vs full CfC fine-tuning on the same data.

# import torch.nn.functional as F
# from aegis.brain.ncp import BRAIN, INPUT_DIM
#
# loader = torch.utils.data.DataLoader(
#     torch.utils.data.TensorDataset(h_states, targets),
#     batch_size=64, shuffle=True
# )
# opt = torch.optim.AdamW(BRAIN.parameters(), lr=5e-5)
# for ep in range(20):
#     for batch_h, batch_t in loader:
#         opt.zero_grad()
#         out = BRAIN(batch_h.unsqueeze(1))
#         loss = F.mse_loss(out.squeeze(1), batch_t)
#         loss.backward()
#         opt.step()
#     print(f"BPTT epoch {ep}: loss={loss.item():.6f}")

## Interpreting Results

| Test MSE | Verdict |
|----------|--------|
| < 0.01 | ✅ Reservoir passes |
| 0.01–0.05 | ⚠️ Marginal |
| > 0.05 | ❌ Reservoir fails |

**Spec results (random-init brain):** ALL should FAIL. If any passes, the
predicate is too loose and must be tightened before this gate is trusted.

**Hard gates:** ablation_brain and ablation_zero must pass (brain output
must differ from zeroed output). If they fail at random init, the
architecture is decorative and the NCP wiring has a structural problem.